<!-- nav -->
[← 8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb) · [Home](../../README.md) · [8.12 · POSIX ERE →](08_12_POSIX_ERE.ipynb)

# 8.11 · POSIX BRE

POSIX defines two grammars of regular expressions (IEEE Std 1003.1, XBD 9): basic ones, which `grep`, `sed` and `ed`
read, and extended ones ([8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb)). A **basic regular expression** writes its operators
with backslashes, `\(a\)` a subexpression and `a\{1,3\}` an interval, its other characters standing for themselves, and
has backreferences, `\1`. It is a dialect of its own, `POSIX BRE`, of the POSIX family, held by its own holder, and read
by C++'s `std::regex` when its flags name the `basic` or the `grep` grammar ([8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb),
08.10.4). grep is no dialect, since POSIX defines no grammar of its own for it: it reads a newline-separated list of BREs,
`pattern_list`, which is a kind of the dialect.

This page builds a BRE from one character up, then its bracket expressions and intervals; then where its anchors may
stand, grep's list of patterns, how C++ reads a BRE, and the editions it covers. It starts from `dialects()`; `S` is its
holder namespace, and `char`, `alternative` and `pattern` write a character and the two kinds every pattern is made of,
as specs the builders call, as on the previous page:

In [1]:
import assert from "node:assert/strict";
import * as CppRegExp from "@mbse/expressions/Cpp/RegExp";
import { Simplifiers, Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import * as BRE from "@mbse/expressions/POSIX/BRE";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const char = (c: string) => (u: any) => u.character((r: any) => r.value(BigInt(c.codePointAt(0)!)));
const alternative = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const pattern = (...alternatives: unknown[]) => (u: any) => u.pattern((r: any) => r.alternatives(alternatives));
const anchor = (kind: string) => (u: any) => u.assertion((r: any) => r.assertionKind(kind));
const DIALECT = BRE.Expressions.DIALECT;

/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(simplify: (expression: any) => unknown, expression: any): string {
  try {
    const result: any = simplify(expression);
    return String(typeof result?.evaluate === "function" ? result.evaluate() : result);
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.11.1"></a>
## 08.11.1 · A BRE, from `ab` to `^\(a\)b*\1$`

```text
ab        ^\(a\)b*\1$
```

`ab` is a `pattern` of one `alternative` of two `character`s, written here in full, as in ECMAScript's grammar: POSIX BRE names its kinds as
`Cpp RegExp` does where the two grammars share a construct, so that a tree reads alike in both. `^\(a\)b*\1$` holds
the rest of a BRE's grammar: anchors, `assertion`s at its start and end; a subexpression, a `capturinggroup`; `b*`, a
`quantifier` of `b`; and `\1`, a `backreference` to the first subexpression. `BRE.Text.ToText` writes each as a BRE
does, its operators escaped:

In [2]:
const backreferenced = pattern(alternative(
  anchor("start"), (u: any) => u.capturinggroup((r: any) => r.alternatives([alternative(char("a"))])),
  (u: any) => u.quantifier((r: any) => r.min(0n).element(char("b"))),
  (u: any) => u.backreference((r: any) => r.ref(1n)), anchor("end")));
const [ab, repeated] = store.transaction(() => [
  S.POSIX.BRE.Expression((e: any) => e.root((u: any) => u.pattern((p: any) => p.alternatives([
    (a: any) => a.alternative((r: any) => r.elements([char("a"), char("b")]))])))),
  S.POSIX.BRE.Expression((e: any) => e.root(backreferenced)),
]);
for (const expression of [ab, repeated]) {
  console.log(BRE.Text.ToText(expression).padEnd(12), [...new Set([...Terms.walk(expression)].map((t) => Terms.kindOf(t)[0]))].join(", "));
}
console.log(DIALECT.name(), DIALECT.tag, "| its kinds:", [...DIALECT.kinds().keys()].join(", "));
assert.deepEqual([ab, repeated].map((x) => BRE.Text.ToText(x)), ["ab", "^\\(a\\)b*\\1$"]);

ab           pattern, alternative, character


^\(a\)b*\1$  pattern, alternative, assertion, capturinggroup, character, quantifier, backreference


POSIX BRE posix_bre | its kinds: patternlist, pattern, alternative, capturinggroup, quantifier, characterclass, characterclassrange, characterset, character, assertion, backreference, exclass, collatingelement, equivalence, error, missing


<a id="T08.11.2"></a>
## 08.11.2 · Bracket expressions and intervals

```text
[[:digit:]a-f]\{1,3\}        [^]a]        [[.ch.][=e=]].
```

A bracket expression is a `characterclass`, negated with `^`, holding characters, ranges (`characterclassrange`) and
POSIX's three bracketed classes: a character class, `[:digit:]` (`exclass`), a collating element, `[.ch.]`
(`collatingelement`), and an equivalence class, `[=e=]` (`equivalence`). An interval, `\{1,3\}`, is a quantifier with a
`max`. `.` is the one `characterset` a BRE has, any character. Within brackets nothing is escaped: `]` first is a
character, and the printer writes it so:

In [3]:
const brackets: any[] = store.transaction(() => [
  [(u: any) => u.quantifier((r: any) => r.min(1n).max(3n).element((u: any) => u.characterclass((c: any) => c.elements([
    (u: any) => u.exclass((x: any) => x.name("digit")),
    (u: any) => u.characterclassrange((g: any) => g.min(char("a")).max(char("f")))]))))],
  [(u: any) => u.characterclass((c: any) => c.negate(true).elements([char("]"), char("a")]))],
  [(u: any) => u.characterclass((c: any) => c.elements([(u: any) => u.collatingelement((x: any) => x.name("ch")),
    (u: any) => u.equivalence((x: any) => x.name("e"))])),
  (u: any) => u.characterset((s: any) => s.setKind("any"))],
].map((spec) => S.POSIX.BRE.Expression((e: any) => e.root(pattern(alternative(...spec))))));
for (const expression of brackets) {
  console.log(BRE.Text.ToText(expression).padEnd(22), [...new Set([...Terms.walk(expression)].map((t) => Terms.kindOf(t)[0]))].join(", "));
}
assert.deepEqual(brackets.map((x) => BRE.Text.ToText(x)), ["[[:digit:]a-f]\\{1,3\\}", "[^]a]", "[[.ch.][=e=]]."]);

[[:digit:]a-f]\{1,3\}  pattern, alternative, quantifier, characterclass, exclass, characterclassrange, character


[^]a]                  pattern, alternative, characterclass, character


[[.ch.][=e=]].         pattern, alternative, characterclass, collatingelement, equivalence, characterset


<a id="T08.11.3"></a>
## 08.11.3 · Where an anchor stands, and what else is checked

A BRE's grammar puts its anchors at its ends: `^` anchors only where a BRE begins, and `$` only where it ends; elsewhere
each is a character, and the dialect, which holds an anchor as an `assertion`, refuses one held anywhere else
(XBD 9.3.8). A backreference is one digit, `\1` to `\9`; a `.` within brackets is a character, not a set; and the
constructs a BRE lacks, ECMAScript's `\d` among them, are refused by name. A BRE binds nothing: its subexpressions are
numbered, as C++'s are:

In [4]:
const checked: Record<string, any> = store.transaction(() => Object.fromEntries(([
  ["a^", [char("a"), anchor("start")]],
  ["$a", [anchor("end"), char("a")]],
  ["\\(^a\\)", [(u: any) => u.capturinggroup((r: any) => r.alternatives([alternative(anchor("start"), char("a"))]))]],
  ["\\10", [(u: any) => u.backreference((r: any) => r.ref(10n))]],
  ["\\d", [(u: any) => u.characterset((s: any) => s.setKind("digit"))]],
  ["^a$", [anchor("start"), char("a"), anchor("end")]],
] as [string, unknown[]][]).map(([source, spec]) => [source, S.POSIX.BRE.Expression((e: any) => e.root(pattern(alternative(...spec))))])));
for (const [source, expression] of Object.entries(checked)) {
  console.log(source.padEnd(7), "|", DIALECT.validate(expression).join("; ") || "valid",
    "| needs:", [...Symbolics.free(expression, DIALECT)].sort().join(", ") || "nothing");
}
assert.deepEqual(DIALECT.validate(checked["a^"]), ["argument 0: a start assertion stands only where a BRE begins"]);

a^      | argument 0: a start assertion stands only where a BRE begins | needs: nothing


$a      | argument 0: an end assertion stands only where a BRE ends | needs: nothing


\(^a\)  | argument 0: argument 0: a BRE's subexpression holds no anchor | needs: nothing


\10     | argument 0: argument 0: a backreference's ref must be 1 to 9, got 10 | needs: nothing


\d      | argument 0: argument 0: CharacterSet.setKind cannot be 'digit' | needs: nothing


^a$     | valid | needs: nothing


<a id="T08.11.4"></a>
## 08.11.4 · grep's list of patterns

```text
grep -e '^a' -e 'b$' -e 'ab\|c'
```

grep reads its patterns as a list, one a line (XCU grep, `pattern_list`), and a line matches when any of them matches.
The list is a kind of the dialect, `patternlist`, whose `patterns` are BREs, each anchored where it begins and ends,
printed a line each. `ab\|c` is a BRE's alternation, `\|`, which POSIX.1-2024 added: the dialect holds it, a `pattern`
of two alternatives, as POSIX's latest edition writes it:

In [5]:
const listed = store.transaction(() => S.POSIX.BRE.Expression((e: any) => e.root((u: any) => u.patternlist((r: any) => r.patterns([
  pattern(alternative(anchor("start"), char("a"))), pattern(alternative(char("b"), anchor("end"))),
  pattern(alternative(char("a"), char("b")), alternative(char("c")))])))));
console.log(BRE.Text.ToText(listed));
console.log(Terms.kindOf(Terms.term(listed))[0], "of", Terms.form(Terms.term(listed)).arguments.length, "patterns |",
  DIALECT.validate(listed).join("; ") || "valid");
assert.equal(BRE.Text.ToText(listed), "^a\nb$\nab\\|c");

^a
b$
ab\|c


patternlist of 3 patterns | valid


<a id="T08.11.5"></a>
## 08.11.5 · Read by C++, and never simplified

```text
std::regex r("^\\(a\\)b*\\1$", std::regex::basic);        std::regex r("^a\nb$", std::regex::grep);
```

A C++ `regex` whose flags name `basic` or `grep` holds a POSIX BRE term as its pattern, and `Cpp RegExp` validates it as
POSIX BRE does, then checks what C++ selects of it ([8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb), 08.10.4): grep's list only
under `grep`, and no alternation, which the POSIX editions C++ cites lack. Nothing simplifies an BRE: matching is a
library's, and the framework's rules fold none of its kinds: each stays in the residual, named by its dialect:

In [6]:
const twoLines = (u: any) => u.patternlist((r: any) => r.patterns([
  pattern(alternative(anchor("start"), char("a"))), pattern(alternative(char("b"), anchor("end")))]));
const regexes: Record<string, any> = store.transaction(() => Object.fromEntries(([
  ["basic: ^\\(a\\)b*\\1$", "basic", backreferenced],
  ["grep: ^a, b$", "grep", twoLines],
  ["basic: ^a, b$", "basic", twoLines],
  ["grep: ab\\|c", "grep", pattern(alternative(char("a"), char("b")), alternative(char("c")))],
] as [string, string, unknown][]).map(([source, flag, spec]) => [source, S.Cpp.RegExp.Expression((e: any) => e.root((u: any) => u.regex(
  (r: any) => r.flags([flag]).pattern((t: any) => t.posix_bre(spec)))))])));
for (const [source, regex] of Object.entries(regexes)) {
  console.log(source.padEnd(20), "|", CppRegExp.Expressions.DIALECT.validate(regex).join("; ") || "valid");
}
const simplify = new Simplifiers.Simplifier(new Simplifiers.Interpreter(DIALECT, new Map(), { folded: new Set() }), { literal: () => null }); // no kind folded
console.log(refused((x) => simplify.run(x), Terms.term(repeated)));
assert.deepEqual(CppRegExp.Expressions.DIALECT.validate(regexes["basic: ^\\(a\\)b*\\1$"]), []);

basic: ^\(a\)b*\1$   | valid


grep: ^a, b$         | valid


basic: ^a, b$        | a patternlist is not in the basic grammar


grep: ab\|c          | alternation is not in the grep grammar


POSIX BRE pattern is not evaluated


<a id="T08.11.6"></a>
## 08.11.6 · Its editions

`POSIX BRE` covers POSIX.1's editions since 2001, IEEE Std 1003.1-2001 to -2024, each by its designation and its common
names; the ISO/IEC designations under which ISO published two of them, which C++'s editions cite, are among those names,
so C++17's reference to ISO/IEC 9945:2003 finds POSIX.1-2001
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [7]:
const bre = store.Instances.Expressions.POSIX.BRE; // the dialect, by its global name
for (const [designation, edition] of Standards.standards(bre)) {
  console.log(designation.padEnd(21), "|", Standards.tags(edition).slice(1).join(", "), "|", Standards.kinds(edition).length, "kinds");
}
assert.ok(Standards.find(bre, "ISO/IEC 9945:2003") === Standards.find(bre, "POSIX.1-2001"));

IEEE Std 1003.1-2001  | POSIX.1-2001, ISO/IEC 9945:2003 | 14 kinds


IEEE Std 1003.1-2008  | POSIX.1-2008, ISO/IEC/IEEE 9945:2009 | 14 kinds


IEEE Std 1003.1-2017  | POSIX.1-2017 | 14 kinds


IEEE Std 1003.1-2024  | POSIX.1-2024 | 14 kinds


## What you learned

- **A BRE is a `POSIX BRE` term**, its kinds named as `Cpp RegExp`'s where the grammars share a construct, printed with
  a BRE's backslashes.
- **Its grammar is checked**: anchors at its ends alone, backreferences of one digit, no construct it lacks.
- **grep's list of patterns is a kind**, `patternlist`, a BRE a line.
- **C++ reads a BRE** under its `basic` and `grep` grammars, checking what it selects of POSIX; nothing simplifies one.
- **Its editions are POSIX.1's**, found by the ISO designations C++ cites too.

Next, POSIX's extended regular expressions, [8.12 · POSIX ERE](08_12_POSIX_ERE.ipynb).

<!-- nav -->
[← 8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb) · [Home](../../README.md) · [8.12 · POSIX ERE →](08_12_POSIX_ERE.ipynb)